# 05 - Inference

This notebook runs the full deblurring pipeline on aerial images using the
crop-based finetuned MambaIR model from `04_training_crop.ipynb`.

```
Input image
     ↓
  YOLOv8 — detect all objects
     ↓
  Crop each object region
     ↓
  MambaIR_finetuned_model — deblur each crop
     ↓
  Paste deblurred crops back into original image
     ↓
  Save result
```

## (Optional) Mount Google Drive

Run this cell only if you are using **Google Colab** and your files are stored on Google Drive.
Skip this cell if you are running locally.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

## Step 1 - Check GPU

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 2 - Install Dependencies

> ⚠️ Required for A100 + CUDA 12.8. Takes ~5 minutes.

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.10.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

Check the versions of Pytorch, Cuda after applied compatibilty

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

 ## Step 3 - Clone MambaIR Repository & Install Dependencies
 

In [ ]:
import os
if not os.path.exists('MambaIR'):
    os.system('git clone https://github.com/csguoh/MambaIR.git')
os.chdir('MambaIR')


In [ ]:
import os
os.system('pip install -q basicsr einops timm')
os.system('pip install -q causal_conv1d==1.0.0')

os.system('pip install -q ultralytics')
os.system('pip install -q -e .')
print('Tüm bağımlılıklar kuruldu.')

In [ ]:
!pip install mamba_ssm --no-build-isolation

In [ ]:
import os
os.system('pip install -q -e .')
print('BasicSR kuruldu.')

## Step 4 - Set Paths

```
# Google Colab + Drive example:
# CHECKPOINT_PATH = '/content/drive/MyDrive/your_project/experiments_crop/models/net_g_10000.pth'
# INPUT_DIR       = '/content/drive/MyDrive/your_project/test_images'
# OUTPUT_DIR      = '/content/drive/MyDrive/your_project/results'

# Local machine example:
# CHECKPOINT_PATH = '/path/to/experiments_crop/models/net_g_10000.pth'
# INPUT_DIR       = '/path/to/test_images'
# OUTPUT_DIR      = '/path/to/results'
```

> `CHECKPOINT_PATH` should point to the best checkpoint from `04_training_crop.ipynb`.
> `INPUT_DIR` can contain images directly (no train/val/test subfolder needed).

In [ ]:
# ← SET YOUR PATHS HERE
CHECKPOINT_PATH = 'checkpoint_path'  # best checkpoint from 04_training_crop.ipynb
INPUT_DIR       = 'input_path'       # folder containing images to deblur
OUTPUT_DIR      = 'output_path'      # folder where results will be saved

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Paths set.')

## Step 5 - Inference Settings

| Parameter | Value | Description |
|-----------|-------|-------------|
| `CONF_THRESH` | 0.05 | YOLO confidence threshold |
| `CROP_PADDING` | 100 | Extra pixels around detected object |
| `MIN_CROP_SIZE` | 128 | Minimum crop size |
| `OUTPUT_SIZE` | 256 | Size fed into deblur model |

> Lower `CONF_THRESH` detects more objects but may include false positives.
> Increase to 0.1–0.3 if too many incorrect detections appear.

In [ ]:
CONF_THRESH   = 0.05
CROP_PADDING  = 100
MIN_CROP_SIZE = 128
OUTPUT_SIZE   = 256

## Step 6 - Load Models

In [ ]:
import torch, sys
sys.path.insert(0, '.')
from ultralytics import YOLO
from basicsr.archs.mambair_arch import MambaIR

def load_deblur_model(ckpt_path):
    model = MambaIR(
        upscale=1, in_chans=3, img_size=128, img_range=1.,
        d_state=16, depths=[6,6,6,6,6,6], embed_dim=180, mlp_ratio=1.2
    )
    state = torch.load(ckpt_path, map_location='cpu')
    model.load_state_dict(state.get('params', state), strict=True)
    model.eval()
    return model.cuda() if torch.cuda.is_available() else model

print('Loading models...')
yolo_model   = YOLO('yolov8n.pt')
deblur_model = load_deblur_model(CHECKPOINT_PATH)
print('Models loaded.')

## Step 7 - Define Pipeline Functions

In [ ]:
import cv2, numpy as np

def get_all_crops(img, result, padding, min_size):
    """Extract crops for all detected objects. Falls back to image center if none found."""
    h, w = img.shape[:2]
    crops = []
    if result.boxes is not None and len(result.boxes) > 0:
        for box in result.boxes.xyxy:
            x1, y1, x2, y2 = box.cpu().numpy()
            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)
            size = max(int(max(x2-x1, y2-y1)) + padding * 2, min_size)
            half = size // 2
            x1c = max(0, cx - half)
            y1c = max(0, cy - half)
            x2c = min(w, x1c + size)
            y2c = min(h, y1c + size)
            if x2c - x1c < size: x1c = max(0, x2c - size)
            if y2c - y1c < size: y1c = max(0, y2c - size)
            crop = img[int(y1c):int(y2c), int(x1c):int(x2c)]
            if crop.size > 0:
                crops.append((crop, int(x1c), int(y1c), int(x2c), int(y2c)))
    else:
        cx, cy = w // 2, h // 2
        half = min_size // 2
        x1c = max(0, cx - half)
        y1c = max(0, cy - half)
        x2c = min(w, x1c + min_size)
        y2c = min(h, y1c + min_size)
        crop = img[y1c:y2c, x1c:x2c]
        if crop.size > 0:
            crops.append((crop, x1c, y1c, x2c, y2c))
    return crops


@torch.no_grad()
def deblur_patch(model, patch_bgr, output_size):
    """Run MambaIR deblurring on a single BGR patch."""
    rgb = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    t   = torch.from_numpy(rgb).permute(2,0,1).unsqueeze(0)
    if torch.cuda.is_available(): t = t.cuda()
    out = model(t).squeeze(0).permute(1,2,0).cpu().numpy()
    out = np.clip(out, 0, 1)
    return cv2.cvtColor((out * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)


def full_pipeline(img_path, yolo_model, deblur_model,
                  conf=0.05, padding=100, min_size=128, output_size=256):
    """Full deblurring pipeline: detect → crop → deblur → paste back."""
    import time
    img = cv2.imread(str(img_path))
    if img is None:
        raise FileNotFoundError(f'Image not found: {img_path}')

    result_img = img.copy()
    h, w = img.shape[:2]

    t0 = time.time()
    results = yolo_model(img, conf=conf, verbose=False)
    result  = results[0]
    t_yolo  = time.time() - t0

    crops  = get_all_crops(img, result, padding, min_size)
    bboxes = []

    t1 = time.time()
    for crop, x1c, y1c, x2c, y2c in crops:
        crop_w = x2c - x1c
        crop_h = y2c - y1c
        crop_resized  = cv2.resize(crop, (output_size, output_size),
                                   interpolation=cv2.INTER_LANCZOS4)
        deblurred_256 = deblur_patch(deblur_model, crop_resized, output_size)
        deblurred     = cv2.resize(deblurred_256, (crop_w, crop_h),
                                   interpolation=cv2.INTER_LANCZOS4)
        result_img[y1c:y2c, x1c:x2c] = deblurred
        bboxes.append((x1c, y1c, x2c, y2c))
    t_deblur = time.time() - t1

    print(f'YOLO    : {t_yolo*1000:.1f} ms')
    print(f'Deblur  : {t_deblur*1000:.1f} ms  ({len(bboxes)} object(s))')
    print(f'Total   : {(t_yolo+t_deblur)*1000:.1f} ms')

    return img, result_img, bboxes

print('Pipeline functions defined.')

## Step 8 - Single Image Test

Test the pipeline on a single image and visualize the result.

> Update `TEST_IMAGE` to point to your test image.

In [ ]:
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

# ← SET YOUR TEST IMAGE PATH
TEST_IMAGE = 'test_image_path'

orig, result, bboxes = full_pipeline(
    TEST_IMAGE, yolo_model, deblur_model,
    conf=CONF_THRESH, padding=CROP_PADDING,
    min_size=MIN_CROP_SIZE, output_size=OUTPUT_SIZE
)

# Draw bounding boxes on original
orig_bbox = cv2.cvtColor(orig.copy(), cv2.COLOR_BGR2RGB)
for (x1, y1, x2, y2) in bboxes:
    cv2.rectangle(orig_bbox, (x1,y1), (x2,y2), (255,0,0), 3)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(cv2.cvtColor(orig,   cv2.COLOR_BGR2RGB))
axes[0].set_title('Original',                    fontsize=13); axes[0].axis('off')
axes[1].imshow(orig_bbox)
axes[1].set_title(f'YOLO Detections ({len(bboxes)} object(s))', fontsize=13); axes[1].axis('off')
axes[2].imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
axes[2].set_title('Deblurred Output',            fontsize=13); axes[2].axis('off')
plt.tight_layout()
plt.savefig('single_result.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'{len(bboxes)} object(s) processed.')

## Step 9 - Batch Inference

Processes all images in `INPUT_DIR` and saves results to `OUTPUT_DIR`.

> Errors on individual images are caught and logged — processing continues.

In [ ]:
from pathlib import Path
from tqdm import tqdm
import cv2

imgs = sorted([f for f in Path(INPUT_DIR).iterdir()
               if f.suffix.lower() in ('.png', '.jpg', '.jpeg')])
print(f'Processing {len(imgs)} images...')

total_objects = 0
for img_path in tqdm(imgs):
    try:
        _, result, bboxes = full_pipeline(
            str(img_path), yolo_model, deblur_model,
            conf=CONF_THRESH, padding=CROP_PADDING,
            min_size=MIN_CROP_SIZE, output_size=OUTPUT_SIZE
        )
        cv2.imwrite(str(Path(OUTPUT_DIR) / img_path.name), result)
        total_objects += len(bboxes)
    except Exception as e:
        print(f'Error on {img_path.name}: {e}')

print(f'Done. {len(imgs)} images processed, {total_objects} objects deblurred.')
print(f'Results saved to: {OUTPUT_DIR}')

## Notes

- YOLOv8n is used for object detection (no custom training required)
- If YOLO finds no object in an image, the center region is used as fallback
- Increase `CONF_THRESH` (e.g. 0.1–0.3) to reduce false positive detections
